In [ ]:
# Running Analysis

# Cada linha refere-se a um segundo
# Posso calcular: cadencia em spm multiplicando a cadencia atual (rpm) por 2
# posso calcular o pace em min/km convertendo o atual que esta em m/s

# 0 - IMPORTS

In [1]:
import fitparse
import pandas as pd
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import plotly.graph_objects as go
pio.renderers.default = 'iframe'

## 0.1 - Helper Functions

In [ ]:
def clean_session(df):
    # 1. Filtra colunas relevantes
    df = df[['distance', 'timestamp', 'cadence', 'heart_rate', 'speed']]
    
    # 2. Interpola NaNs no meio
    df = df.interpolate(method='linear')
    
    # 3. Remove linhas iniciais ainda com NaN
    df = df.dropna(subset=['distance', 'timestamp', 'cadence', 'heart_rate', 'speed'])
    
    # 4. Reset index
    df = df.reset_index(drop=True)
    
    return df

def feature_engineering(df):
    # 1. Criando novas features
    df['year']  = df['timestamp'].dt.year
    df['month'] = df['timestamp'].dt.month
    df['day']   = df['timestamp'].dt.day
    df['week']  = df['timestamp'].dt.isocalendar().week
    df['time']  = df['timestamp'].dt.time
    df['pace']  = (1000 / (df['speed']*60)).round(2)
    df['spm']   = df['cadence']*2
    df['session'] = 0
    df['training_type'] = 0

    # Dropando a coluna timestamp
    df = df.drop(columns=['timestamp'])
    
    return df

# 1 - DATA MANIPULATION

## 1.1 - Loading Data

In [ ]:
fitfile = fitparse.FitFile('../data/6952866dce3bf01f7b3508bc.fit')

## 1.2 - Data Processing

### 1.2.1 - Data Extraction

In [ ]:
records = []
for record in fitfile.get_messages('record'):
    data={}
    for field in record:
        data[field.name] = field.value
    records.append(data)
    
df = pd.DataFrame(records)

### 1.2.2 - Data Dimensions

In [ ]:
# Data Dimensions
print( 'The dataset contains {}'.format( df.shape[0] ),'rows.' )
print( 'The dataset contains {}'.format( df.shape[1] ),'columns.' )

### 1.2.3 - Data Types

In [ ]:
df.dtypes

### 1.2.4 - Selecting relevant columns

In [ ]:
df = df[['distance', 'timestamp', 'cadence', 'heart_rate', 'speed']]

### 1.2.5 - Checking Duplicates

In [ ]:
# Check Duplicates
df.duplicated().sum()

### 1.2.6 - Checking NaN

In [ ]:
# Check NA
df.isna().sum()

In [ ]:
df[df.isna().any(axis=1)]

#### 1.2.6.1 - Handling NaN Values

Going to remove the firsts rows with NaN values

In [ ]:
# Removing first rows where distance = 0. Meaning I did not start to run yet.
df = df.dropna(subset=['distance', 'timestamp', 'cadence', 'heart_rate', 'speed'])

In [ ]:
# Filling any NaN value in the middle with similar data.
df = df.interpolate(method='linear')

In [ ]:
# Data Dimensions
print( 'The dataset contains {}'.format( df.shape[0] ),'rows.' )
print( 'The dataset contains {}'.format( df.shape[1] ),'columns.' )

## 1.3 - FEATURE ENGINEERING

In [ ]:
df['year']  = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['day']   = df['timestamp'].dt.day
df['week']  = df['timestamp'].dt.isocalendar().week
df['time']  = df['timestamp'].dt.time
df['pace']  = 1000 / (df['speed']*60).round(2)
df['spm']   = df['cadence']*2
df['session'] = 0
df['training_type'] = 0

In [ ]:
def session_summary(df):
    duration = len(df)  # cada linha = 1 segundo
    duration_min = duration / 60

    print(f"Duração:       {duration_min:.1f} min")
    print(f"Distância:     {df['distance'].max() / 1000:.2f} km")
    print(f"Pace médio:    {df['pace'].mean():.2f} min/km")
    print(f"Pace melhor:   {df['pace'].min():.2f} min/km")
    print(f"FC média:      {df['heart_rate'].mean():.0f} bpm")
    print(f"FC máxima:     {df['heart_rate'].max():.0f} bpm")
    print(f"Cadência média:{df['spm'].mean():.0f} spm")

In [ ]:
session_summary(df)

In [ ]:
fig = px.line(df, x='time', y='heart_rate', title='FC ao longo do treino')
fig.show()

In [ ]:
from matplotlib.collections import LineCollection
import matplotlib.pyplot as plt
import numpy as np

fig, ax1 = plt.subplots()

# FC por zonas
lc = LineCollection(segments, colors=colors, linewidth=2)
ax1.add_collection(lc)
ax1.autoscale()
ax1.set_ylabel('BPM')

# Pace no eixo secundário
ax2 = ax1.twinx()
ax2.plot(range(len(df)), df['pace'], color='purple', linewidth=1.5, label='Pace')
ax2.invert_yaxis()
ax2.set_ylabel('min/km')

plt.title('FC por Zona e Pace')
plt.show()